# Automatic Circuit Discovery

**Circuit discovery** identifies the minimal subset of model components responsible for a specific behavior. Instead of manually inspecting each head, we can systematically ablate components and measure their effect.

This notebook covers:
1. **Sweep ablations**: Test every head's importance
2. **Find circuit**: Automatically identify the minimal circuit
3. **Attention pattern analysis**: Entropy, focus, similarity
4. **Causal tracing**: Corrupt a token and measure recovery
5. **Batch experiments**: Run experiments across many prompts

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk.experiments import (
    run_on_dataset, sweep_ablations, find_circuit,
    compare_metrics, ExperimentResult,
)
from irtk.attention_utils import (
    entropy, all_head_entropy, head_pattern_similarity,
    attention_to_token, top_attended_tokens,
    causal_tracing, attention_head_summary,
)
from irtk.patching import make_logit_diff_metric
from irtk import vis

print("Loading GPT-2 Small...")
model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Loaded: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Setting Up a Task

We'll study a simple factual recall task: "The Eiffel Tower is in" -> " Paris".
We define a metric and collect a small dataset.

In [ ]:
# Define our prompts and expected answers
prompts_and_answers = [
    ("The Eiffel Tower is located in the city of", " Paris"),
    ("The Colosseum is located in the city of", " Rome"),
    ("The Statue of Liberty is located in the city of", " New"),
    ("Big Ben is located in the city of", " London"),
]

# Tokenize and set up metrics
dataset = []
for prompt, answer in prompts_and_answers:
    tokens = model.to_tokens(prompt)
    answer_id = tokenizer.encode(answer)[0]
    logits = model(tokens)
    pred = tokenizer.decode([int(jnp.argmax(logits[-1]))])
    print(f"{prompt!r} -> predicted: {pred!r}, expected: {answer!r} (token {answer_id})")
    dataset.append((tokens, answer_id))

# Simple metric: logit of the correct answer token
def make_answer_logit_metric(answer_token):
    def metric(logits):
        return float(logits[-1, answer_token])
    return metric

## 2. Sweep Ablations

For each head, zero-ablate it and measure the effect on the target logit. Heads whose ablation causes a big drop are important for the task.

In [ ]:
# Use the first prompt for a quick demo
demo_tokens, demo_answer = dataset[0]
demo_metric = make_answer_logit_metric(demo_answer)

# Sweep head ablations
result = sweep_ablations(model, [demo_tokens], demo_metric, method="zero", component="heads")
baseline = float(model(demo_tokens)[-1, demo_answer])

# Plot: how does each head's ablation change the metric?
effect = result.values - baseline  # negative = ablation hurts, positive = helps

fig, ax = vis.plot_head_summary(
    effect,
    title=f"Ablation Effect on '{prompts_and_answers[0][1].strip()}' logit",
    cmap="RdBu_r",
    vmin=-2, vmax=2,
)
plt.show()

print("Top 5 most important heads (ablation hurts most):")
flat = effect.flatten()
for i in np.argsort(flat)[:5]:
    l = i // model.cfg.n_heads
    h = i % model.cfg.n_heads
    print(f"  L{l}H{h}: {effect[l,h]:+.3f}")

## 3. Automatic Circuit Discovery

`find_circuit()` automates the process: it ablates each head, computes the relative metric change, and identifies heads above a threshold.

In [ ]:
# Automatic circuit discovery
circuit_result = find_circuit(
    model,
    [demo_tokens],
    demo_metric,
    threshold=0.05,  # heads that change the metric by >5%
    method="zero",
)

circuit_heads = circuit_result.values["circuit_heads"]
baseline_val = circuit_result.values["baseline"]

print(f"Baseline metric: {baseline_val:.3f}")
print(f"Found {len(circuit_heads)} circuit heads (threshold=5%):")
for layer, head, importance in circuit_heads:
    print(f"  L{layer}H{head}: {importance:.1%} relative change")

# Visualize the circuit
rel_change = circuit_result.values["relative_change"]
fig, ax = vis.plot_head_summary(
    rel_change,
    title="Relative Metric Change per Head (circuit = above threshold)",
    cmap="Reds",
)
plt.show()

## 4. Attention Pattern Analysis

Once we've identified important heads, we can analyze their attention patterns to understand **what** they do.

In [ ]:
# Get cache for attention pattern analysis
_, cache = model.run_with_cache(demo_tokens)

# Attention entropy for all heads
ent = all_head_entropy(cache, model)
fig, ax = vis.plot_head_summary(
    ent,
    title="Attention Entropy (high = diffuse, low = focused)",
    cmap="viridis",
)
plt.show()

# Full summary
summary = attention_head_summary(cache, model)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

vis.plot_head_summary(summary["max_attn"], title="Mean of Max Attention", cmap="viridis", ax=ax1)
vis.plot_head_summary(summary["diag_score"], title="Self-Attention Score", cmap="viridis", ax=ax2)

plt.tight_layout()
plt.show()

In [ ]:
# Look at the attention patterns of the top circuit heads
token_labels = [tokenizer.decode([t]) for t in np.array(demo_tokens)]

top_heads = circuit_heads[:4] if len(circuit_heads) >= 4 else circuit_heads
if top_heads:
    fig, axes = plt.subplots(1, len(top_heads), figsize=(5 * len(top_heads), 5))
    if len(top_heads) == 1:
        axes = [axes]
    
    for ax, (l, h, imp) in zip(axes, top_heads):
        pattern = np.array(cache[("pattern", l)][h])
        im = ax.imshow(pattern, cmap='viridis', aspect='auto')
        ax.set_title(f"L{l}H{h} ({imp:.0%})")
        if len(token_labels) <= 15:
            ax.set_xticks(range(len(token_labels)))
            ax.set_xticklabels(token_labels, rotation=90, fontsize=6)
            ax.set_yticks(range(len(token_labels)))
            ax.set_yticklabels(token_labels, fontsize=6)
        plt.colorbar(im, ax=ax, fraction=0.046)
    
    plt.suptitle("Attention Patterns of Top Circuit Heads", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # What does the last position attend to?
    print("\nTop attended tokens from the last position:")
    for l, h, imp in top_heads:
        top = top_attended_tokens(cache, l, h, query_pos=-1, k=5)
        top_strs = [f"{token_labels[pos]}({weight:.2f})" for pos, weight in top]
        print(f"  L{l}H{h}: {', '.join(top_strs)}")

## 5. Causal Tracing

**Causal tracing** (from the ROME paper, Meng et al. 2022) corrupts a specific token's embedding and then restores clean activations at each layer. This reveals where the model stores and processes the information from that token.

In [ ]:
# Causal tracing: corrupt "France" and see where the info is recovered
prompt_ct = "The Eiffel Tower is located in the city of"
tokens_ct = model.to_tokens(prompt_ct)
labels_ct = [tokenizer.decode([t]) for t in np.array(tokens_ct)]

# Find the subject token ("Eiffel" or similar key token)
print("Tokens:", labels_ct)

# Corrupt the subject token and trace recovery
# Let's corrupt position 1 (typically a key content word)
subject_pos = 1  # Adjust based on tokenization
print(f"Corrupting token at position {subject_pos}: {labels_ct[subject_pos]!r}")

trace = causal_tracing(
    model, tokens_ct,
    corrupt_pos=subject_pos,
    metric_fn=demo_metric,
    noise_std=3.0,
)

print(f"\nClean metric: {trace['clean']:.3f}")
print(f"Corrupted metric: {trace['corrupted']:.3f}")

# Plot recovery curves
fig, ax = plt.subplots(figsize=(10, 5))
layers = range(model.cfg.n_layers)

ax.plot(layers, trace['restored_resid'], 'o-', label='Restore residual', alpha=0.8)
ax.plot(layers, trace['restored_attn'], 's-', label='Restore attention', alpha=0.8)
ax.plot(layers, trace['restored_mlp'], '^-', label='Restore MLP', alpha=0.8)
ax.axhline(y=trace['clean'], color='green', linestyle='--', alpha=0.5, label='Clean')
ax.axhline(y=trace['corrupted'], color='red', linestyle='--', alpha=0.5, label='Corrupted')

ax.set_xlabel('Layer')
ax.set_ylabel('Metric Value')
ax.set_title(f'Causal Tracing: Corrupting {labels_ct[subject_pos]!r}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Batch Experiments Across Prompts

To get robust results, we should run experiments across multiple prompts. The `experiments` module makes this easy.

In [ ]:
# Run ablation sweep across all prompts
all_tokens = [tokens for tokens, _ in dataset]
all_metrics = [make_answer_logit_metric(ans) for _, ans in dataset]

# Average the ablation effect across all prompts
all_effects = []
for tokens, answer in dataset:
    metric = make_answer_logit_metric(answer)
    result = sweep_ablations(model, [tokens], metric, component="heads")
    baseline_i = float(model(tokens)[-1, answer])
    all_effects.append(result.values - baseline_i)

avg_effect = np.mean(all_effects, axis=0)

fig, ax = vis.plot_head_summary(
    avg_effect,
    title=f"Average Ablation Effect Across {len(dataset)} Prompts",
    cmap="RdBu_r",
    vmin=-1.5, vmax=1.5,
)
plt.show()

print("Heads consistently important across prompts:")
flat = avg_effect.flatten()
for i in np.argsort(flat)[:5]:
    l = i // model.cfg.n_heads
    h = i % model.cfg.n_heads
    print(f"  L{l}H{h}: avg effect {avg_effect[l,h]:+.3f}")

## Key Takeaways

1. **Sweep ablations** systematically test every head's importance for a task.

2. **`find_circuit()`** automates circuit discovery by thresholding ablation effects.

3. **Attention entropy** helps classify heads: low entropy = focused attention (often important), high entropy = diffuse (often less specific).

4. **Causal tracing** reveals which layers store/process information from specific tokens.

5. **Averaging across prompts** gives more robust circuits than single-prompt analysis.

### API Reference

```python
from irtk.experiments import (
    run_on_dataset,          # Run metric across many prompts
    sweep_ablations,         # Systematically ablate heads/layers
    find_circuit,            # Auto-discover minimal circuit
    compare_metrics,         # Run multiple metrics on same dataset
    ExperimentResult,        # Structured result container
)

from irtk.attention_utils import (
    entropy,                     # Per-position attention entropy
    all_head_entropy,            # [n_layers, n_heads] mean entropy
    head_pattern_similarity,     # Cosine sim between two heads
    attention_to_token,          # Attention flowing to a key position
    top_attended_tokens,         # Top-k attended positions
    causal_tracing,              # Corrupt + restore experiment
    attention_head_summary,      # Summary stats for all heads
)
```